In [1]:
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text

SERVER = r"DANNYPC\SQLEXPRESS"
DATABASE = "MAG7_MarketDB"
DRIVER = "ODBC Driver 17 for SQL Server"
COMPANY_INFO_FILE = r"C:\Users\Admin\Desktop\Project\data\company_info\mag7_company_info.csv"

params = urllib.parse.quote_plus(
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

print("Connected to:", pd.read_sql("SELECT DB_NAME() AS db_name", engine).iloc[0, 0])

df = pd.read_csv(COMPANY_INFO_FILE)

out = df[[
    "ticker",
    "short_name",
    "current_ceo",
    "sector",
    "industry",
    "long_business_summary",
    "address1",
    "city",
    "state",
    "zip",
    "country",
    "full_time_employees",
    "current_price",
    "market_cap",
    "enterprise_value"
]].copy()

with engine.begin() as conn:
    conn.execute(text("DELETE FROM dbo.company_info"))

out.to_sql(
    "company_info",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False,
    chunksize=100
)

print(f"Loaded {len(out)} rows into dbo.company_info.")
print("Done.")

Connected to: MAG7_MarketDB
Loaded 7 rows into dbo.company_info.
Done.
